In [1]:
import torch.optim as optim

from src.training import get_accuracy, OneHotEncode
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

In [2]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [3]:
from src.improved_model import BinarizingCNN

# Instantiate the model
model = BinarizingCNN().to(device)
model.set_scramble_distance(0.05)
#criterion = nn.CrossEntropyLoss()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=float(1e-2))

In [ ]:
# Training loop
num_epochs: int = 500
target_accuracy: float = .96
maximum_scramble_distance: float = 5.0

test_data, _ = next(iter(test_dataloader))
test_data.to(device)


for epoch in range(num_epochs):
    for inputs, labels in train_dataloader:
        if epoch == 0:
            break
        # Forward pass
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.layer2.scale.data.clamp_(min=1.0)
        model.layer3.scale.data.clamp_(min=1.0)

    # Assess progress:
    validation_accuracy: float = get_accuracy(model, val_dataloader)
    if validation_accuracy > target_accuracy:
        model.set_scramble_distance(min(model.scramble_distance + .1, maximum_scramble_distance))
        print(f"Scramble distance: {model.scramble_distance:.2f}")
    print(f'Epoch [{epoch+1}/{num_epochs}], Accuracy: {validation_accuracy:.4f}')


Epoch [1/500], Accuracy: 0.0965
Epoch [2/500], Accuracy: 0.9046
Epoch [3/500], Accuracy: 0.9352
Epoch [4/500], Accuracy: 0.9490
Epoch [5/500], Accuracy: 0.9547
Epoch [6/500], Accuracy: 0.9563
Epoch [7/500], Accuracy: 0.9597
Scramble distance: 0.15
Epoch [8/500], Accuracy: 0.9611
Epoch [9/500], Accuracy: 0.9523
Epoch [10/500], Accuracy: 0.9542
Scramble distance: 0.25
Epoch [11/500], Accuracy: 0.9601
Epoch [12/500], Accuracy: 0.9521
Epoch [13/500], Accuracy: 0.9555
Epoch [14/500], Accuracy: 0.9552
Epoch [15/500], Accuracy: 0.9573
Epoch [16/500], Accuracy: 0.9592
Epoch [17/500], Accuracy: 0.9597
Scramble distance: 0.35
Epoch [18/500], Accuracy: 0.9615
Epoch [19/500], Accuracy: 0.9537
Epoch [20/500], Accuracy: 0.9548
Epoch [21/500], Accuracy: 0.9573
Epoch [22/500], Accuracy: 0.9593
Epoch [23/500], Accuracy: 0.9588
Scramble distance: 0.45
Epoch [24/500], Accuracy: 0.9604
Epoch [25/500], Accuracy: 0.9519
Epoch [26/500], Accuracy: 0.9569
Epoch [27/500], Accuracy: 0.9562
Epoch [28/500], Accura

In [ ]:
from src.improved_model import BinarizingNetwork


def get_signed_accuracy(model: BinarizingNetwork, dataloader: DataLoader) -> float:
    model.eval_mode()

    with torch.no_grad():  # Disable gradient computation
        all_correct: int = 0
        for inputs, labels in dataloader:
            # Move inputs and labels to the specified device
            inputs, labels = inputs.to(device), labels.to(device)
            outputs: torch.Tensor = model(inputs)
            comparison: torch.Tensor = torch.argmax(outputs, axis=1) == torch.argmax(
                labels, axis=1
            )
            all_correct += sum(comparison)
        accuracy: float = all_correct / len(dataloader.dataset)
    model.train_mode()
    return accuracy
test_data = test_data.to(device)
model.eval_mode()
model(test_data[0:1])

d = test_data[0:1]
#print(model.float_to_binary_layer(d))

print(model.layer3.bias)
#
# # model(test_data[0,...])
print(get_signed_accuracy(model, val_dataloader))
